In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.compose import make_column_transformer
from sklearn.pipeline import Pipeline
from sklearn.pipeline import make_pipeline
import re
import joblib 

In [2]:
df = pd.read_csv(r"F:\practice\DataScience_project_Pradyumna pradhan\house price prediction\Bengaluru_House_Data (1).csv")
df.head()

,area_type,availability,location,size,society,total_sqft,bath,balcony,price
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,Coomee,1056,2.0,1.0,39.07
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,Theanmp,2600,5.0,3.0,120.00
2,Built-up Area,Ready To Move,Uttarahalli,3 BHK,NaN,1440,2.0,3.0,62.00
3,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,Soiewre,1521,3.0,1.0,95.00
4,Super built-up Area,Ready To Move,Kothanur,2 BHK,NaN,1200,2.0,1.0,51.00


In [3]:
unique_unit = set()

for val in df["total_sqft"]:
    val = str(val)
    if any(c.isalpha() for c in val):
        unique_unit.add(val)
unique_unit

{'1.25Acres',
 '1.26Acres',
 '1000Sq. Meter',
 '1100Sq. Meter',
 '1100Sq. Yards',
 '117Sq. Yards',
 '120Sq. Yards',
 '122Sq. Yards',
 '132Sq. Yards',
 '133.3Sq. Yards',
 '142.61Sq. Meter',
 '142.84Sq. Meter',
 '1500Cents',
 '1500Sq. Meter',
 '151.11Sq. Yards',
 '1574Sq. Yards',
 '15Acres',
 '167Sq. Meter',
 '188.89Sq. Yards',
 '1Grounds',
 '2.09Acres',
 '204Sq. Meter',
 '24Guntha',
 '24Sq. Meter',
 '2940Sq. Yards',
 '2Acres',
 '300Sq. Yards',
 '3040Sq. Meter',
 '30Acres',
 '315Sq. Yards',
 '34.46Sq. Meter',
 '361.33Sq. Yards',
 '38Guntha',
 '3Cents',
 '4125Perch',
 '45.06Sq. Meter',
 '45Sq. Yards',
 '5.31Acres',
 '500Sq. Yards',
 '697Sq. Meter',
 '6Acres',
 '716Sq. Meter',
 '78.03Sq. Meter',
 '84.53Sq. Meter',
 '86.72Sq. Meter'}

In [4]:
import re

In [4]:
Unit_name = set()

for val in df["total_sqft"]:
    val = str(val).lower()
    words = re.findall(r"[a-zA-Z]+",val)
    for w in words:
        Unit_name.add(w)
Unit_name

{'acres', 'cents', 'grounds', 'guntha', 'meter', 'perch', 'sq', 'yards'}

In [5]:
def convert_sqft_to_num(x):
    try:
        import re
        x = str(x).strip()

        # case 1: range like "1000-1200"
        if '-' in x:
            tokens = x.split('-')
            if len(tokens) == 2:
                try:
                    return (float(tokens[0]) + float(tokens[1])) / 2
                except ValueError:
                    return None
            else:
                return None   

        # case 2: contains letters (unit attached or separated like "100 perch" or "100perch")
        elif any(c.isalpha() for c in x):
            # Use regex to split number and unit, handling attached units like '100perch'
            match = re.match(r'([0-9.]+)\s*([a-zA-Z.]+)', x.lower().replace(',', ''))
            if match:
                value, unit = match.groups()
                try:
                    value = float(value)
                except ValueError:
                    return None
                if 'meter' in unit:
                    return value * 10.7639
                elif 'yard' in unit:
                    return value * 0.836127
                elif 'acre' in unit:
                    return value * 4046.86
                elif 'cent' in unit:
                    return value * 40.4686
                elif 'guntha' in unit:
                    return value * 101.1714
                elif 'perch' in unit:
                    return value * 25.2929
                elif 'ground' in unit:
                    return value * 203.0
                elif 'sq' in unit:
                    return value  # assume sq.ft
                else:
                    return None
            else:
                return None
        # case 3: just a number
        else:
            try:
                return float(x)
            except ValueError:
                return None
    except:
        return None

In [6]:
df["total_sqft_cleaned"] = df["total_sqft"].apply(convert_sqft_to_num)

In [7]:
df[["total_sqft","total_sqft_cleaned"]].head()

,total_sqft,total_sqft_cleaned
0,1056,1056.0
1,2600,2600.0
2,1440,1440.0
3,1521,1521.0
4,1200,1200.0


In [8]:
df["total_sqft_cleaned"].unique

<bound method Series.unique of 0        1056.0
1        2600.0
2        1440.0
3        1521.0
4        1200.0
          ...  
13315    3453.0
13316    3600.0
13317    1141.0
13318    4689.0
13319     550.0
Name: total_sqft_cleaned, Length: 13320, dtype: float64>

In [9]:
df

,area_type,availability,location,size,society,total_sqft,bath,balcony,price,total_sqft_cleaned
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,Coomee,1056,2.0,1.0,39.07,1056.0
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,Theanmp,2600,5.0,3.0,120.00,2600.0
2,Built-up Area,Ready To Move,Uttarahalli,3 BHK,NaN,1440,2.0,3.0,62.00,1440.0
3,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,Soiewre,1521,3.0,1.0,95.00,1521.0
4,Super built-up Area,Ready To Move,Kothanur,2 BHK,NaN,1200,2.0,1.0,51.00,1200.0
...,...,...,...,...,...,...,...,...,...,...
13315,Built-up Area,Ready To Move,Whitefield,5 Bedroom,ArsiaEx,3453,4.0,0.0,231.00,3453.0
13316,Super built-up Area,Ready To Move,Richards Town,4 BHK,NaN,3600,5.0,NaN,400.00,3600.0
13317,Built-up Area,Ready To Move,Raja Rajeshwari Nagar,2 BHK,Mahla T,1141,2.0,1.0,60.00,1141.0
13318,Super built-up Area,18-Jun,Padmanabhanagar,4 BHK,SollyCl,4689,4.0,1.0,488.00,4689.0


In [10]:
df.isnull().sum()

area_type                0
availability             0
location                 1
size                    16
society               5502
total_sqft               0
bath                    73
balcony                609
price                    0
total_sqft_cleaned       0
dtype: int64

In [11]:
df['size'] = df['size'].fillna('2 BHK')

In [12]:
df['location'] = df['location'].fillna('Whitefield')

In [13]:
# Some entries may be like '1 BHK' or 'Super built-up 1 BHK' but typical dataset uses formats like '2 BHK' or '4 Bedroom'
df['bhk'] = df['size'].str.split().str.get(0).astype(int)

In [14]:
df['bath'] = df['bath'].fillna(df['bath'].median())

In [15]:
# price per sqrft
df['price_per_sqft'] = df['price'] * 100000 / df['total_sqft_cleaned']

In [16]:
df['price_per_sqft'].describe()

count    1.332000e+04
mean     8.091836e+03
std      1.065841e+05
min      2.429867e+01
25%      4.267260e+03
50%      5.440000e+03
75%      7.333333e+03
max      1.200000e+07
Name: price_per_sqft, dtype: float64

In [17]:
df.describe()

,bath,balcony,price,total_sqft_cleaned,bhk,price_per_sqft
count,13320.000000,12711.000000,13320.000000,13320.000000,13320.000000,1.332000e+04
mean,2.688814,1.584376,112.565627,1587.538313,2.802778,8.091836e+03
std,1.338754,0.817263,148.971674,2001.524080,1.294496,1.065841e+05
min,1.000000,0.000000,8.000000,1.000000,1.000000,2.429867e+01
25%,2.000000,1.000000,50.000000,1100.000000,2.000000,4.267260e+03
50%,2.000000,2.000000,72.000000,1275.000000,3.000000,5.440000e+03
75%,3.000000,2.000000,120.000000,1680.000000,3.000000,7.333333e+03
max,40.000000,3.000000,3600.000000,121405.800000,43.000000,1.200000e+07


In [18]:
# price in lakhs
df['price_in_lakhs'] = df['price'] * 100000

In [19]:
df['price_in_lakhs']

0         3907000.0
1        12000000.0
2         6200000.0
3         9500000.0
4         5100000.0
            ...    
13315    23100000.0
13316    40000000.0
13317     6000000.0
13318    48800000.0
13319     1700000.0
Name: price_in_lakhs, Length: 13320, dtype: float64

In [20]:
df['location'].value_counts()

location
Whitefield                 541
Sarjapur  Road             399
Electronic City            302
Kanakpura Road             273
Thanisandra                234
                          ... 
Park View Layout             1
Xavier Layout                1
Air View Colony              1
akshaya nagar t c palya      1
mvj engineering college      1
Name: count, Length: 1305, dtype: int64

In [21]:
df['location'] = df['location'].apply(lambda x: x.strip())

In [22]:
df['location']

0        Electronic City Phase II
1                Chikka Tirupathi
2                     Uttarahalli
3              Lingadheeranahalli
4                        Kothanur
                   ...           
13315                  Whitefield
13316               Richards Town
13317       Raja Rajeshwari Nagar
13318             Padmanabhanagar
13319                Doddathoguru
Name: location, Length: 13320, dtype: object

In [ ]:
# remove outlier 
#df = df[(df['total_sqft_cleaned'] / df['bhk']) > 300]


In [23]:
def remove_outlier_sqft(df):
    df_output = pd.DataFrame()
    for key,subdf in df.groupby('location'):
        m = np.mean(subdf.price_per_sqft)
        
        st = np.std(subdf.price_per_sqft)
        
        gen_df = subdf[(subdf.price_per_sqft > (m-st)) & (subdf.price_per_sqft <= (m+st))]
        
        df_output = pd.concat([df_output,gen_df], ignore_index= True)
    return df_output
df = remove_outlier_sqft(df)
df.describe()


,bath,balcony,price,total_sqft_cleaned,bhk,price_per_sqft,price_in_lakhs
count,10337.000000,9957.000000,10337.000000,10337.000000,10337.000000,10337.000000,1.033700e+04
mean,2.534585,1.587727,98.654149,1520.750461,2.644384,6070.598555,9.865415e+06
std,1.068923,0.798289,116.422820,1103.600405,0.996353,7320.952933,1.164228e+07
min,1.000000,0.000000,8.000000,11.000000,1.000000,357.478802,8.000000e+05
25%,2.000000,1.000000,49.000000,1100.000000,2.000000,4255.319149,4.900000e+06
50%,2.000000,2.000000,68.000000,1280.000000,3.000000,5258.964143,6.800000e+06
75%,3.000000,2.000000,105.000000,1651.000000,3.000000,6666.666667,1.050000e+07
max,14.000000,3.000000,2912.000000,60702.900000,10.000000,672727.272727,2.912000e+08


In [24]:
def bhk_outlier_remover(df):
    exclude_indices = np.array([])
    for location, location_df in df.groupby('location'):
        bhk_stats = {}
        for bhk, bhk_df in df.groupby('bhk'):
            bhk_stats[bhk] ={
                'mean': np.mean(bhk_df.price_per_sqft),
                'std': np.std(bhk_df.price_per_sqft),
                'count': bhk_df.shape[0]
            }
        for bhk, bhk_df in location_df.groupby('bhk'):
            stats = bhk_stats.get(bhk-1)
            if stats and stats['count']>5:
                exclude_indices = np.append(exclude_indices, bhk_df[bhk_df.price_per_sqft<(stats['mean'])].index.values)
    return df.drop(exclude_indices, axis='index')


In [25]:
df = bhk_outlier_remover(df)

In [26]:
df

,area_type,availability,location,size,society,total_sqft,bath,balcony,price,total_sqft_cleaned,bhk,price_per_sqft,price_in_lakhs
0,Super built-up Area,Ready To Move,1st Block BEL Layout,3 BHK,NaN,1540,3.0,2.0,85.00,1540.0,3,5519.480519,8500000.0
1,Super built-up Area,Ready To Move,1st Block HBR Layout,1 BHK,NaN,600,1.0,0.0,45.00,600.0,1,7500.000000,4500000.0
3,Plot Area,18-Feb,1st Block HRBR Layout,8 Bedroom,NaN,1200,7.0,0.0,235.00,1200.0,8,19583.333333,23500000.0
4,Plot Area,Ready To Move,1st Block HRBR Layout,7 Bedroom,NaN,2400,7.0,3.0,355.00,2400.0,7,14791.666667,35500000.0
5,Plot Area,Ready To Move,1st Block HRBR Layout,3 Bedroom,NaN,600,3.0,1.0,90.00,600.0,3,15000.000000,9000000.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
10329,Super built-up Area,Ready To Move,Yeshwanthpur,3 BHK,IBityin,1675,3.0,NaN,92.13,1675.0,3,5500.298507,9213000.0
10330,Super built-up Area,Ready To Move,Yeshwanthpur Industrial Suburb,6 BHK,Essic C,3800,6.0,NaN,390.00,3800.0,6,10263.157895,39000000.0
10331,Super built-up Area,Ready To Move,frazertown,3 BHK,NaN,2900,3.0,2.0,325.00,2900.0,3,11206.896552,32500000.0
10333,Plot Area,Ready To Move,south,3 Bedroom,NaN,2400,2.0,0.0,480.00,2400.0,3,20000.000000,48000000.0


In [27]:
df.to_csv("new_bengaluru_data5.csv", index=False)

In [28]:
df

,area_type,availability,location,size,society,total_sqft,bath,balcony,price,total_sqft_cleaned,bhk,price_per_sqft,price_in_lakhs
0,Super built-up Area,Ready To Move,1st Block BEL Layout,3 BHK,NaN,1540,3.0,2.0,85.00,1540.0,3,5519.480519,8500000.0
1,Super built-up Area,Ready To Move,1st Block HBR Layout,1 BHK,NaN,600,1.0,0.0,45.00,600.0,1,7500.000000,4500000.0
3,Plot Area,18-Feb,1st Block HRBR Layout,8 Bedroom,NaN,1200,7.0,0.0,235.00,1200.0,8,19583.333333,23500000.0
4,Plot Area,Ready To Move,1st Block HRBR Layout,7 Bedroom,NaN,2400,7.0,3.0,355.00,2400.0,7,14791.666667,35500000.0
5,Plot Area,Ready To Move,1st Block HRBR Layout,3 Bedroom,NaN,600,3.0,1.0,90.00,600.0,3,15000.000000,9000000.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
10329,Super built-up Area,Ready To Move,Yeshwanthpur,3 BHK,IBityin,1675,3.0,NaN,92.13,1675.0,3,5500.298507,9213000.0
10330,Super built-up Area,Ready To Move,Yeshwanthpur Industrial Suburb,6 BHK,Essic C,3800,6.0,NaN,390.00,3800.0,6,10263.157895,39000000.0
10331,Super built-up Area,Ready To Move,frazertown,3 BHK,NaN,2900,3.0,2.0,325.00,2900.0,3,11206.896552,32500000.0
10333,Plot Area,Ready To Move,south,3 Bedroom,NaN,2400,2.0,0.0,480.00,2400.0,3,20000.000000,48000000.0


In [29]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5054 entries, 0 to 10336
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   area_type           5054 non-null   object 
 1   availability        5054 non-null   object 
 2   location            5054 non-null   object 
 3   size                5054 non-null   object 
 4   society             3440 non-null   object 
 5   total_sqft          5054 non-null   object 
 6   bath                5054 non-null   float64
 7   balcony             4811 non-null   float64
 8   price               5054 non-null   float64
 9   total_sqft_cleaned  5054 non-null   float64
 10  bhk                 5054 non-null   int64  
 11  price_per_sqft      5054 non-null   float64
 12  price_in_lakhs      5054 non-null   float64
dtypes: float64(6), int64(1), object(6)
memory usage: 552.8+ KB


In [30]:
df["total_sqft_cleaned"].isnull().sum()


np.int64(0)

In [31]:
# Check which rows have null values in total_sqft_cleaned
null_rows = df[df["total_sqft_cleaned"].isnull()][["total_sqft_cleaned"]]
print(f"Number of null values: {len(null_rows)}")
print("\nRows with null values in total_sqft_cleaned:")
print(null_rows)

Number of null values: 0

Rows with null values in total_sqft_cleaned:
Empty DataFrame
Columns: [total_sqft_cleaned]
Index: []


In [32]:
# take requied vcolumns
df = df[['location','price','total_sqft_cleaned', 'bhk']]

In [33]:
df

,location,price,total_sqft_cleaned,bhk
0,1st Block BEL Layout,85.00,1540.0,3
1,1st Block HBR Layout,45.00,600.0,1
3,1st Block HRBR Layout,235.00,1200.0,8
4,1st Block HRBR Layout,355.00,2400.0,7
5,1st Block HRBR Layout,90.00,600.0,3
...,...,...,...,...
10329,Yeshwanthpur,92.13,1675.0,3
10330,Yeshwanthpur Industrial Suburb,390.00,3800.0,6
10331,frazertown,325.00,2900.0,3
10333,south,480.00,2400.0,3


In [34]:
# 4. Feature selection
x = df.drop('price', axis=1)
y = df['price']

In [35]:
# 5. Train-test split
x_train,x_test,y_train,y_test = train_test_split(x, y, test_size=0.2, random_state=0 )

In [36]:
print(x_train.shape)
print(x_test.shape)

(4043, 3)
(1011, 3)


apply linear regression

In [37]:
col_trans = make_column_transformer(
    (OneHotEncoder(sparse_output=False, handle_unknown='ignore'), ['location']),
    remainder='passthrough'
)

In [38]:
scaler = StandardScaler()
lr = LinearRegression()
pipe = make_pipeline(col_trans,scaler,lr)
pipe.fit(x_train,y_train)

,steps,"[('columntransformer', ...), ('standardscaler', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('onehotencoder', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [39]:
y_pred_lr = pipe.predict(x_test)
r2_score(y_test,y_pred_lr)

0.803854441016558

Apply Lasso

In [40]:
lasso = Lasso()
pipe = make_pipeline(col_trans,scaler,lasso)
pipe.fit(x_train,y_train)

,steps,"[('columntransformer', ...), ('standardscaler', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('onehotencoder', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [41]:
y_pred_lasso = pipe.predict(x_test)
r2_score(y_test,y_pred_lasso)

0.8034994534299805

Apply Ridge

In [42]:
ridge = Ridge()
pipe = make_pipeline(col_trans, scaler, ridge)
pipe.fit(x_train,y_train)

,steps,"[('columntransformer', ...), ('standardscaler', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('onehotencoder', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [43]:
y_pred_ridge = pipe.predict(x_test)
r2_score(y_test,y_pred_ridge)

0.8038576110817555

In [44]:
print("No Regularization:",r2_score(y_test,y_pred_lr))
print("Lasso:",r2_score(y_test,y_pred_lasso))
print("Ridge:",r2_score(y_test,y_pred_ridge))

No Regularization: 0.803854441016558
Lasso: 0.8034994534299805
Ridge: 0.8038576110817555


In [45]:
import pickle

In [46]:
pickle.dump(pipe, open('Ridgemodel.pkl', 'wb'))